# 21：13データセットを順番にTabDiffで学習

共通設定の後、`df_main` から `df_survival` まで各専用セルで学習します。「すべて実行」で連続処理します。通常モデル・EMA・前処理情報・学習グラフはモデルごとに保存されます。

`MAX_TRAIN_ROWS = "all"` は各データセットの全件です。整数指定が対象件数を超える場合や20例未満の場合は停止します。各セルの再実行は継続学習ではなく、最初から新しいモデルを学習します。

19で各学習実行から生成し、20で評価してください。独立した生成表同士を行番号で結合しないでください。

19・20で既存どおり実行を選択できるよう、学習資産の保存先は `artifacts/18_tabdiff/` を維持します。


In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, json, subprocess, hashlib, time
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import matplotlib.pyplot as plt

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
# 下の専用セルを上から実行すると、DATASET_NAMESの順で個別モデルを学習します。
DATASET_NAME = "df_main"
DATASET_NAMES = ['df_main', 'df_witness', 'df_bystandercpr', 'df_bystandercpr_ventilation', 'df_bystandercpr_aed', 'df_defibrillation', 'df_airway', 'df_airway_airwaydevice', 'df_adrenaline', 'df_cardiac', 'df_noncardiac', 'df_rosc', 'df_survival']
if DATASET_NAME not in DATASET_NAMES:
    raise ValueError(f"対象は次から選択してください: {DATASET_NAMES}")
PROCESSED_DIR = PROJECT / "data/processed"
INPUT = PROCESSED_DIR / ("utstein_" + DATASET_NAME.removeprefix("df_") + ".parquet")

TABDIFF_REPO = PROJECT / "external/TabDiff"  # 既存の公式コードがあれば変更
TABDIFF_PYTHON = Path(sys.executable)       # TabDiff環境のPython
# ===== 学習条件：実行前に空欄へ数値を入力してください =====
# 症例数："all"で対象サブセットの全件。整数なら指定件数を抽出して90%/10%分割します。
MAX_TRAIN_ROWS = "all"          # 例: 100_000（"all"なら全症例）
EPOCHS = "200"                  # 例: 200
TRAIN_BATCH_SIZE = "4096"        # 例: 1024（GPUメモリ不足なら小さくします）
SAMPLE_BATCH_SIZE = "512"       # 例: 512（学習後の生成用）
DIFFUSION_STEPS = "50"         # 例: 50（拡散の時間ステップ数）
CHECKPOINT_SELECTION = "ema"    # "ema" または "normal"（最終エポックのEMA/通常重み）
N_GENERATE = 10_000           # 19で生成する件数の初期値

GPU = 0                       # CPUなら-1（大規模データは非常に時間がかかります）
LOSS_EVAL_EVERY = 5          # 固定評価データで損失を測る間隔（epoch）
LOSS_EVAL_ROWS = 2048        # 学習・ホールドアウトそれぞれの評価上限（全件評価はしません）
LOSS_EVAL_BATCH_SIZE = 512  # 評価用バッチサイズ
SEED = 42
EMA_DECAY = 0.997             # 各バッチでEMAを更新


def _required_positive_int(name, value):
    """空欄を許さず、正の整数として学習設定を検証します。"""
    if value == "" or value is None:
        raise ValueError(f"{name} が空欄です。上の学習条件に正の整数を入力してください。")
    try:
        number = int(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{name} は整数で入力してください: {value!r}") from exc
    if number <= 0:
        raise ValueError(f"{name} は1以上にしてください。")
    return number

USE_ALL_ROWS = str(MAX_TRAIN_ROWS).strip().lower() == "all"
if not USE_ALL_ROWS:
    MAX_TRAIN_ROWS = _required_positive_int("症例数 MAX_TRAIN_ROWS", MAX_TRAIN_ROWS)
EPOCHS = _required_positive_int("エポック数 EPOCHS", EPOCHS)
TRAIN_BATCH_SIZE = _required_positive_int("学習バッチサイズ TRAIN_BATCH_SIZE", TRAIN_BATCH_SIZE)
SAMPLE_BATCH_SIZE = _required_positive_int("生成バッチサイズ SAMPLE_BATCH_SIZE", SAMPLE_BATCH_SIZE)
DIFFUSION_STEPS = _required_positive_int("拡散ステップ数 DIFFUSION_STEPS", DIFFUSION_STEPS)
N_GENERATE = _required_positive_int("生成件数 N_GENERATE", N_GENERATE)
LOSS_EVAL_EVERY = _required_positive_int("LOSS_EVAL_EVERY", LOSS_EVAL_EVERY)
LOSS_EVAL_ROWS = _required_positive_int("LOSS_EVAL_ROWS", LOSS_EVAL_ROWS)
LOSS_EVAL_BATCH_SIZE = _required_positive_int("LOSS_EVAL_BATCH_SIZE", LOSS_EVAL_BATCH_SIZE)
CHECKPOINT_SELECTION = str(CHECKPOINT_SELECTION).strip().lower()
if CHECKPOINT_SELECTION not in {"ema", "normal"}:
    raise ValueError('CHECKPOINT_SELECTION は "ema" または "normal" を入力してください。')

EXCLUDE_COLUMNS = ["ID", "id", "患者ID", "患者番号", "症例ID"]
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
DATA_NAME = "utstein_" + DATASET_NAME.removeprefix("df_") + "_" + RUN_ID
EXP_NAME = "unconditional"
OUT = PROJECT / "artifacts/18_tabdiff" / RUN_ID
assert INPUT.is_file(), f"入力がありません: {INPUT}"
assert (TABDIFF_REPO / "main.py").is_file(), f"公式TabDiffのパスを指定してください: {TABDIFF_REPO}"
OUT.mkdir(parents=True, exist_ok=False)
print("入力:", INPUT, "\n学習出力:", OUT)
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024**2), b''):
            h.update(block)
    return h.hexdigest()

REQUESTED_TRAIN_ROWS = MAX_TRAIN_ROWS
REQUESTED_USE_ALL_ROWS = USE_ALL_ROWS


In [ ]:
# 損失は小さいほどよい指標ですが、生成データの正確さを直接表す「正解率」ではありません。
def make_learning_figure(history):
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    online = [("loss/total_loss", "Train online"), ("ema_loss/total_loss", "EMA train evaluation")]
    panels = [
        (axes[0, 0], "Online training / full-train EMA loss", online),
        (axes[0, 1], "Fixed subset: total loss", [(f"eval/{s}/{m}/total", f"{s} {m}") for s in ("train", "holdout") for m in ("normal", "ema")]),
        (axes[1, 0], "Fixed subset: numerical loss", [(f"eval/{s}/{m}/continuous", f"{s} {m}") for s in ("train", "holdout") for m in ("normal", "ema")]),
        (axes[1, 1], "Fixed subset: categorical loss", [(f"eval/{s}/{m}/categorical", f"{s} {m}") for s in ("train", "holdout") for m in ("normal", "ema")]),
    ]
    for ax, title, lines in panels:
        for key, label in lines:
            if key in history:
                values = pd.to_numeric(history[key], errors="coerce")
                good = values.notna()
                if good.any():
                    ax.plot(history.loc[good, "epoch"], values[good], label=label,
                            linestyle="--" if "holdout" in label else "-", marker=".", markersize=4)
        ax.set(title=title, xlabel="Epoch", ylabel="Loss (lower is better)")
        ax.grid(alpha=.25)
        if ax.lines:
            ax.legend(fontsize=8)
    fig.suptitle("TabDiff learning curves: loss is not synthetic-data accuracy")
    fig.tight_layout()
    return fig

# 標準出力・進捗バー・警告はログファイルへ保存します。
# 画面の表示は同じ場所を更新し、ログ行を大量に増やしません。
from collections import deque

def format_duration(seconds):
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, sec = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{sec:02d}"

def run_command(args, log_name, show_timing=False):
    env = os.environ.copy()
    env.update(WANDB_MODE="disabled", PYTHONUNBUFFERED="1", PYTHONHASHSEED=str(SEED))
    log_path = OUT / log_name
    progress_path = OUT / "training_progress.json"
    if show_timing:
        progress_path.unlink(missing_ok=True)
        (OUT / "training_history.jsonl").unlink(missing_ok=True)
    started = time.monotonic()
    handle = None
    graph_handle = None
    last_graph_epoch = None
    def update_graph():
        nonlocal graph_handle, last_graph_epoch
        history_path = OUT / "training_history.jsonl"
        if not history_path.is_file():
            return
        # 末尾が書き込み途中でも、完了したJSON行だけを利用します。
        rows = []
        for line in history_path.read_text(encoding="utf-8").splitlines():
            try:
                rows.append(json.loads(line))
            except ValueError:
                continue
        if not rows or rows[-1]["epoch"] == last_graph_epoch:
            return
        history = pd.DataFrame(rows)
        last_graph_epoch = int(history["epoch"].iloc[-1])
        fig = make_learning_figure(history)
        if graph_handle is None:
            graph_handle = display(fig, display_id=True)
        else:
            graph_handle.update(fig)
        fig.savefig(OUT / "training_learning_curves.png", dpi=160, bbox_inches="tight")
        history.to_csv(OUT / "training_history.csv", index=False, encoding="utf-8-sig")
        plt.close(fig)
    def update_display(finished=False):
        nonlocal handle
        elapsed = time.monotonic() - started
        estimate = None
        if finished:
            estimate = elapsed
        elif progress_path.is_file():
            try:
                progress = json.loads(progress_path.read_text(encoding="utf-8"))
                done = progress.get("completed_epochs", 0)
                if done > 0:
                    # 起動・前処理時間も含む総経過時間を完了epoch数で外挿。
                    estimate = elapsed + progress["training_elapsed"] / done * max(0, EPOCHS-done)
            except (OSError, ValueError, KeyError):
                pass
        text = ("現在の経過時間：" + format_duration(elapsed)
                + "\n予想総所要時間：" + (format_duration(estimate) if estimate is not None else "推定中（最初の1エポック完了後に表示）"))
        if handle is None:
            handle = display(text, display_id=True)
        else:
            handle.update(text)
        update_graph()
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(list(map(str, args)), cwd=TABDIFF_REPO, env=env,
                                stdout=log, stderr=subprocess.STDOUT)
        try:
            if show_timing:
                update_display()
            last_update = time.monotonic()
            while proc.poll() is None:
                time.sleep(1)
                if show_timing and time.monotonic()-last_update >= 5:
                    update_display()
                    last_update = time.monotonic()
        except KeyboardInterrupt:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            print(f"中断しました。ログ: {log_path}")
            raise
    if proc.returncode:
        with log_path.open(encoding="utf-8", errors="replace") as f:
            print("".join(deque(f, maxlen=80)))
        raise RuntimeError(f"処理失敗。ログ: {log_path}")
    if show_timing:
        update_display(finished=True)

# GPU環境確認の詳細もenvironment.logへ保存します。
run_command([TABDIFF_PYTHON, "-c",
    "import torch,pandas,numpy,sklearn,sdmetrics,wandb,tomli,tomli_w; "
    "print('torch',torch.__version__);print('CUDA',torch.cuda.is_available()); "
    "print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"], "environment.log")
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=TABDIFF_REPO, text=True).strip()


## データセット別の連続学習
共通設定を全モデルに適用します。以下の各セルで1モデルずつ学習します。失敗・中断時はそこで停止し、原因を修正後、そのセルから再実行できます。


In [ ]:
# 各データセットを独立した実行フォルダで、順番に学習します。
# データセット間でモデル・重み・前処理は引き継ぎません。
TRAINING_STEPS = ['real = pd.read_parquet(INPUT, engine="pyarrow")\nif not isinstance(real, pd.DataFrame) or not real.columns.is_unique:\n    raise ValueError("入力は列名が一意のDataFrameにしてください。")\nreal = real.drop(columns=[c for c in EXCLUDE_COLUMNS if c in real.columns]).copy()\n# 01が保存した列順・抽出条件を確認して、別データセットの混入を防ぎます。\nmanifest_path = PROCESSED_DIR / "utstein_datasets_manifest.json"\n# 一覧JSONがあれば照合し、ない場合は01と同じ列・抽出条件を検証します。\nif manifest_path.is_file():\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n    entries = [entry for entry in manifest if entry["dataset"] == DATASET_NAME]\n    if len(entries) != 1:\n        raise ValueError("データセット一覧に対象が一意に存在しません。01の保存セルを再実行してください。")\n    dataset_entry = entries[0]\nelse:\n    main_columns = [\'覚知ウツタイン\', \'都道府県名\', \'性別\', \'年齢\', \'救急救命士乗車\', \'医師の乗車\', \'医師の２次救命処置\', \'目撃\', \'バイスタンダーCPR\', \'口頭指示あり\', \'波形種別\', \'除細動\', \'気道確保\', \'静脈路確保\', \'薬剤投与\', \'心原性非心原性\', \'心拍再開\', \'1ヶ月予後回答\', \'1ヶ月生存\', \'覚知現着分\', \'現着接触分\', \'接触病着分\', \'覚知CPR開始分\']\n    subset_specs = {\'df_main\': ([], []), \'df_witness\': ([(\'目撃\', \'目撃あり\')], [\'覚知目撃分\']), \'df_bystandercpr\': ([(\'バイスタンダーCPR\', \'あり\')], [\'バイスタンダー種別\', \'人工呼吸\', \'バイスタンダーcpraed実施除細動適応\', \'覚知バイスタンダーCPR開始分\']), \'df_bystandercpr_ventilation\': ([(\'バイスタンダーCPR\', \'あり\'), (\'人工呼吸\', \'チェックあり\')], [\'バイスタンダー種別\', \'人工呼吸\', \'バイスタンダーcpraed実施除細動適応\', \'覚知バイスタンダーCPR開始分\', \'覚知バイスタンダー人工呼吸開始分\']), \'df_bystandercpr_aed\': ([(\'バイスタンダーCPR\', \'あり\'), (\'バイスタンダーcpraed実施除細動適応\', \'__AED_POSITIVE__\')], [\'バイスタンダー種別\', \'人工呼吸\', \'バイスタンダーcpraed実施除細動適応\', \'バイスタンダーによる除細動\', \'覚知バイスタンダーAED開始分\']), \'df_defibrillation\': ([(\'除細動\', \'あり\')], [\'二相性単相性\', \'施行回数\', \'実施者救急救命士\', \'実施者救急隊員\', \'実施者消防隊員\', \'実施者その他\', \'覚知初回除細動分\']), \'df_airway\': ([(\'気道確保\', \'あり\')], [\'特定行為器具使用\']), \'df_airway_airwaydevice\': ([(\'気道確保\', \'あり\'), (\'特定行為器具使用\', \'あり\')], [\'特定行為器具使用\', \'特定行為器具種別\', \'覚知特定行為器具挿入分\']), \'df_adrenaline\': ([(\'薬剤投与\', \'あり\')], [\'アドレナリン投与回数\', \'覚知薬剤投与分\']), \'df_cardiac\': ([(\'心原性非心原性\', \'心原性\')], [\'心原性の種別\']), \'df_noncardiac\': ([(\'心原性非心原性\', \'非心原性\')], [\'非心原性の種別\']), \'df_rosc\': ([(\'心拍再開\', \'あり\')], [\'覚知初回心拍再開分\']), \'df_survival\': ([(\'1ヶ月生存\', \'生存あり\')], [\'1ヶ月後のCPC\', \'1ヶ月後のOPC\'])}\n    conditions, extra_columns = subset_specs[DATASET_NAME]\n    # 01と同じ設定。AED適応のコードは原票の定義に合わせてください。\n    aed_positive_value = str(globals().get("AED_POSITIVE_VALUE", "1"))\n    dataset_entry = {"dataset":DATASET_NAME,"file":INPUT.name,"rows":len(real),\n                     "columns":main_columns + extra_columns,"conditions":conditions,\n                     "aed_positive_value":aed_positive_value}\n    print("一覧JSONがないため、組み込みの列順・抽出条件で検証します。")\n    print("一覧JSONの復旧には01の最後の保存セルを再実行してください。")\nif dataset_entry["file"] != INPUT.name or dataset_entry["columns"] != real.columns.tolist() or dataset_entry["rows"] != len(real):\n    raise ValueError("入力と01のデータセット一覧が一致しません。")\nfor column, value in dataset_entry["conditions"]:\n    target = dataset_entry["aed_positive_value"] if value == "__AED_POSITIVE__" else value\n    if not real[column].astype("string").eq(target).fillna(False).all():\n        raise ValueError(f"抽出条件に一致しない行があります: {column}={target}")\nINPUT_SHA256 = sha256_file(INPUT)\nINPUT_ROWS = len(real)\nif USE_ALL_ROWS:\n    MAX_TRAIN_ROWS = INPUT_ROWS\nprint(f"対象: {DATASET_NAME} / 元データ {INPUT_ROWS:,}件 / 抽出予定 {MAX_TRAIN_ROWS:,}件")\n\nif MAX_TRAIN_ROWS > len(real):\n    raise ValueError(f"指定症例数 {MAX_TRAIN_ROWS:,} が元データ件数 {len(real):,} を超えています。")\nreal = real.sample(n=MAX_TRAIN_ROWS, random_state=SEED)\nreal = real.reset_index(drop=True)\nif len(real) < 20:\n    raise ValueError("少なくとも20行必要です。")\nrng = np.random.default_rng(SEED)\nindices = rng.permutation(len(real))\ncut = int(len(real) * .9)\ntrain_idx, test_idx = indices[:cut], indices[cut:]\nschema, encoded = {}, pd.DataFrame(index=real.index)\nconstants = {}\nnum_names, cat_names = [], []\nfor i, column in enumerate(real.columns):\n    series = real[column]\n    key = f"f{i:03d}"\n    # 日付のみのobject列にも対応\n    is_date = pd.api.types.is_datetime64_any_dtype(series) or column in ["発生年月日ウツタイン", "覚知ウツタイン"]\n    if series.nunique(dropna=False) <= 1:\n        constants[column] = series.iloc[0]\n        continue\n    spec = {"key": key, "dtype": series.dtype}\n    if is_date or (pd.api.types.is_numeric_dtype(series) and not isinstance(series.dtype, pd.CategoricalDtype)):\n        if is_date:\n            dt = pd.to_datetime(series, errors="coerce")\n            values = (dt - pd.Timestamp("1970-01-01")).dt.total_seconds() / 86400\n            spec["kind"] = "datetime"\n        else:\n            values = pd.to_numeric(series, errors="coerce").astype(float)\n            spec["kind"] = "numeric"\n        values = values.replace([np.inf, -np.inf], np.nan)\n        observed = values.loc[train_idx].dropna()\n        if observed.empty:\n            raise ValueError(f"学習側に有効な数値がありません: {column}")\n        spec.update(median=float(observed.median()), minimum=float(observed.min()),\n                    maximum=float(observed.max()), integer=bool((observed % 1 == 0).all()))\n        encoded[key] = values.fillna(spec["median"])\n        num_names.append(key)\n        if values.isna().any():\n            spec["mask"] = key + "_missing"\n            encoded[spec["mask"]] = np.where(values.isna(), "missing_yes", "missing_no")\n            cat_names.append(spec["mask"])\n    else:\n        spec["kind"] = "category"\n        labels = series.astype("string")\n        unique = labels.dropna().unique().tolist()\n        if len(unique) > 2000:\n            raise ValueError(f"カテゴリーが多すぎます。ID等でないか確認してください: {column} ({len(unique)})")\n        mapping = {value: f"label_{j:05d}" for j, value in enumerate(unique)}\n        spec["decode"] = {v: k for k, v in mapping.items()}\n        encoded[key] = labels.map(mapping).fillna("missing_category")\n        cat_names.append(key)\n    schema[column] = spec\nif not num_names:\n    raise ValueError("このノートブックでは少なくとも1つの可変数値列が必要です。")\n# 公式実装のtarget欄に数値列1つを指定します。目的変数を条件固定せず全列を同時生成します。\ntarget_key = num_names[0]\nmetadata = {"name": DATA_NAME, "task_type": "regression", "header": "infer",\n    "column_names": encoded.columns.tolist(), "file_type": "csv",\n    "num_col_idx": [encoded.columns.get_loc(c) for c in num_names if c != target_key],\n    "cat_col_idx": [encoded.columns.get_loc(c) for c in cat_names],\n    "target_col_idx": [encoded.columns.get_loc(target_key)],\n    "data_path": f"data/{DATA_NAME}/{DATA_NAME}.csv",\n    "test_path": f"data/{DATA_NAME}/holdout.csv", "val_path": None}\nDATA_DIR = TABDIFF_REPO / "data" / DATA_NAME\nDATA_DIR.mkdir(parents=True, exist_ok=False)\n(TABDIFF_REPO / "data/Info").mkdir(parents=True, exist_ok=True)\nencoded.iloc[train_idx].to_csv(DATA_DIR / f"{DATA_NAME}.csv", index=False)\nencoded.iloc[test_idx].to_csv(DATA_DIR / "holdout.csv", index=False)\n(TABDIFF_REPO / "data/Info" / f"{DATA_NAME}.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")\npd.to_pickle({"schema": schema, "constants": constants, "columns": real.columns.tolist(),\n              "train_idx": train_idx, "test_idx": test_idx, "dtypes": real.dtypes.to_dict(), "dataset_name": DATASET_NAME, "dataset_conditions": dataset_entry["conditions"]}, OUT / "preprocessing.pkl")\ndisplay(pd.DataFrame({"列名": list(schema), "変換": [s["kind"] for s in schema.values()]}))\nprint("抽出症例数:", len(real), "学習:",len(train_idx),"ホールドアウト:",len(test_idx),"モデル入力列数:",encoded.shape[1])', '# 公式スクリプトでNumPy配列とinfo.jsonを生成します。\nrun_command([TABDIFF_PYTHON, "process_dataset.py", "--dataname", DATA_NAME], "preprocess.log")\n# 同じ前処理情報・学習/ホールドアウト表の組み合わせを記録します。\ntraining_assets = {"dataset_name": DATASET_NAME, "input_path": str(INPUT),\n    "input_sha256": INPUT_SHA256, "input_rows": INPUT_ROWS,\n    "dataset_conditions": dataset_entry["conditions"],\n    "aed_positive_value": dataset_entry["aed_positive_value"],\n    \n    "preprocessing_sha256": sha256_file(OUT / "preprocessing.pkl"),\n    "train_csv_sha256": sha256_file(DATA_DIR / f"{DATA_NAME}.csv"),\n    "holdout_csv_sha256": sha256_file(DATA_DIR / "holdout.csv"),\n    "ema_update": "per_batch", "ema_decay": EMA_DECAY,\n    "selected_checkpoint": CHECKPOINT_SELECTION, "epochs": EPOCHS,\n    "train_batch_size": TRAIN_BATCH_SIZE, "diffusion_steps": DIFFUSION_STEPS,\n    "requested_rows": MAX_TRAIN_ROWS,\n    "dataname": DATA_NAME, "exp_name": EXP_NAME,\n}\n(OUT / "training_assets.json").write_text(json.dumps(training_assets, indent=2), encoding="utf-8")\n', '# 専用ラッパー内で互換処理と時間計測を行います。公式ソースは変更しません。\nrunner_source = "\\nimport os, sys, json, argparse, importlib, time, math\\nfrom pathlib import Path\\nimport torch\\ncfg = json.loads(Path(sys.argv[1]).read_text())\\nsys.path.insert(0, cfg[\'repo\'])\\nos.chdir(cfg[\'repo\'])\\n# tqdmをクラスとして保持したまま無効化し、関連ライブラリとの互換性も保ちます。\\nimport tqdm\\n_OriginalTqdm = tqdm.tqdm\\nclass QuietTqdm(_OriginalTqdm):\\n    def __init__(self, *args, **kwargs):\\n        kwargs[\'disable\'] = True\\n        super().__init__(*args, **kwargs)\\ntqdm.tqdm = QuietTqdm\\nmodule = importlib.import_module(\'tabdiff.main\')\\n# PyTorchの版による引数の違いを吸収します（公式ファイルは変更しません）。\\n# verbose非対応の版でのみ引数を除去し、他の引数はそのまま渡します。\\nimport inspect\\ntrainer_module = importlib.import_module(\'tabdiff.trainer\')\\n_original_plateau = trainer_module.ReduceLROnPlateau\\nif \'verbose\' not in inspect.signature(_original_plateau).parameters:\\n    def compatible_plateau(*args, **kwargs):\\n        kwargs.pop(\'verbose\', None)\\n        return _original_plateau(*args, **kwargs)\\n    trainer_module.ReduceLROnPlateau = compatible_plateau\\n    print(\'PyTorch compatibility: ReduceLROnPlateau verbose argument omitted\')\\n\\nTrainer = module.Trainer\\n\\n# バッチ学習の直後にEMA更新。公式側のエポック末更新は無効化します。\\n_original_update_ema = trainer_module.update_ema\\n_original_run_step = Trainer._run_step\\n\\ndef run_step_with_batch_ema(self, *args, **kwargs):\\n    result = _original_run_step(self, *args, **kwargs)\\n    _original_update_ema(self.ema_model.parameters(),\\n                         self.diffusion._denoise_fn.parameters(), rate=self.ema_decay)\\n    _original_update_ema(self.ema_num_schedule.parameters(),\\n                         self.diffusion.num_schedule.parameters(), rate=self.ema_decay)\\n    _original_update_ema(self.ema_cat_schedule.parameters(),\\n                         self.diffusion.cat_schedule.parameters(), rate=self.ema_decay)\\n    return result\\n\\nTrainer._run_step = run_step_with_batch_ema\\n\\ndef skip_epoch_ema(*args, **kwargs):\\n    pass\\ntrainer_module.update_ema = skip_epoch_ema\\n\\noriginal_load = module.src.load_config\\noriginal_loop = Trainer.run_loop\\n\\ndef configured_load(path):\\n    value = original_load(path)\\n    value[\'train\'][\'main\'].update(steps=cfg[\'epochs\'], batch_size=cfg[\'batch\'],\\n                                check_val_every=cfg[\'epochs\'] + 1, ema_decay=cfg[\'ema_decay\'])\\n    value[\'diffusion_params\'][\'num_timesteps\'] = cfg[\'diffusion_steps\']\\n    value[\'sample\'][\'batch_size\'] = cfg[\'sample_batch\']\\n    return value\\nmodule.src.load_config = configured_load\\n\\n# 公式run_loopのEMA全学習表評価結果を記録（公式版によってloggerへ渡されないため捕捉）。\\n_original_compute_loss = Trainer.compute_loss\\ndef compute_loss_with_history(self, *args, **kwargs):\\n    result = _original_compute_loss(self, *args, **kwargs)\\n    self._history_fulltrain_ema_loss = result\\n    return result\\nTrainer.compute_loss = compute_loss_with_history\\n\\ndef fixed_subset_loss(trainer, dataset, seed):\\n    # 評価の乱数を固定し、学習側の乱数状態は復元します。\\n    # 同じデータと同じノイズで通常/EMA・epoch間を比較する近似的な診断です。\\n    from torch.utils.data import DataLoader, Subset\\n    generator = torch.Generator().manual_seed(cfg.get(\'seed\', 42))\\n    indices = torch.randperm(len(dataset), generator=generator)[:cfg[\'loss_eval_rows\']].tolist()\\n    loader = DataLoader(Subset(dataset, indices), batch_size=cfg[\'loss_eval_batch\'], shuffle=False)\\n    device = torch.device(trainer.device)\\n    devices = [device.index if device.index is not None else torch.cuda.current_device()] if device.type == \'cuda\' else []\\n    total_d, total_c, count = 0.0, 0.0, 0\\n    was_training = trainer.diffusion.training\\n    try:\\n        with torch.random.fork_rng(devices=devices):\\n            torch.manual_seed(seed)\\n            trainer.diffusion.eval()\\n            with torch.no_grad():\\n                for batch in loader:\\n                    x = batch.float().to(trainer.device)\\n                    dloss, closs = trainer.diffusion.mixed_loss(x)\\n                    total_d += float(dloss.item()) * len(x)\\n                    total_c += float(closs.item()) * len(x)\\n                    count += len(x)\\n    finally:\\n        trainer.diffusion.train(was_training)\\n    if not count:\\n        return {\'categorical\': None, \'continuous\': None, \'total\': None, \'rows\': 0}\\n    return {\'categorical\': total_d/count, \'continuous\': total_c/count,\\n            \'total\': (total_d+total_c)/count, \'rows\': count}\\n\\nclass TimingLogger:\\n    def __init__(self, logger, started, trainer):\\n        self.logger, self.started, self.trainer = logger, started, trainer\\n    def __getattr__(self, name):\\n        return getattr(self.logger, name)\\n    def log(self, data, *args, **kwargs):\\n        result = self.logger.log(data, *args, **kwargs)\\n        if \'epoch\' in data:\\n            epoch = int(data[\'epoch\'])\\n            row = {\'epoch\': epoch, \'elapsed_seconds\': time.monotonic()-self.started}\\n            # EMAのcompute_lossは公式実装では学習データ上の損失です。検証損失と呼びません。\\n            for key in (\'lr\', \'loss/c_loss\', \'loss/d_loss\', \'loss/total_loss\',\\n                        \'ema_loss/c_loss\', \'ema_loss/d_loss\', \'ema_loss/total_loss\'):\\n                if key in data:\\n                    row[key] = float(data[key])\\n            if hasattr(self.trainer, \'_history_fulltrain_ema_loss\'):\\n                dloss, closs = self.trainer._history_fulltrain_ema_loss\\n                row.update({\'ema_loss/d_loss\':float(dloss), \'ema_loss/c_loss\':float(closs),\\n                            \'ema_loss/total_loss\':float(dloss+closs)})\\n            if epoch == 1 or epoch % cfg[\'loss_eval_every\'] == 0 or epoch == cfg[\'epochs\']:\\n                trainer = self.trainer\\n                for split, dataset in ((\'train\', trainer.dataset), (\'holdout\', trainer.test_dataset)):\\n                    metrics = fixed_subset_loss(trainer, dataset, cfg.get(\'seed\',42)+101)\\n                    row.update({f\'eval/{split}/normal/{key}\':value for key,value in metrics.items()})\\n                current = trainer.to_ema_model()\\n                try:\\n                    for split, dataset in ((\'train\', trainer.dataset), (\'holdout\', trainer.test_dataset)):\\n                        metrics = fixed_subset_loss(trainer, dataset, cfg.get(\'seed\',42)+101)\\n                        row.update({f\'eval/{split}/ema/{key}\':value for key,value in metrics.items()})\\n                finally:\\n                    trainer.to_model(*current)\\n            row = {key:(None if isinstance(value,float) and not math.isfinite(value) else value) for key,value in row.items()}\\n            with Path(cfg[\'history_path\']).open(\'a\', encoding=\'utf-8\') as handle:\\n                handle.write(json.dumps(row, allow_nan=False) + \'\\\\n\')\\n            progress = {\'completed_epochs\': epoch, \'training_elapsed\': time.monotonic()-self.started}\\n            target = Path(cfg[\'progress_path\'])\\n            temp = target.with_suffix(\'.tmp\')\\n            temp.write_text(json.dumps(progress), encoding=\'utf-8\')\\n            os.replace(temp, target)\\n        return result\\n\\ndef train_and_save(self):\\n    self.logger = TimingLogger(self.logger, time.monotonic(), self)\\n    original_loop(self)\\n    save_dir = Path(self.model_save_path)\\n    normal_path = save_dir / (\'model_final_%d.pt\' % self.curr_epoch)\\n    ema_path = save_dir / (\'final_ema_%d.pt\' % self.curr_epoch)\\n    # 書き込み途中のファイルをモデル一覧に載せないよう、一時ファイルから置換。\\n    states = [\\n        (normal_path, {\'denoise_fn\': self.diffusion._denoise_fn.state_dict(),\\n                      \'num_schedule\': self.diffusion.num_schedule.state_dict(),\\n                      \'cat_schedule\': self.diffusion.cat_schedule.state_dict()}),\\n        (ema_path, {\'denoise_fn\': self.ema_model.state_dict(),\\n                   \'num_schedule\': self.ema_num_schedule.state_dict(),\\n                   \'cat_schedule\': self.ema_cat_schedule.state_dict()}),\\n    ]\\n    for path, weights in states:\\n        temp = path.with_suffix(\'.tmp\')\\n        torch.save(weights, temp)\\n        os.replace(temp, path)\\n    selected_path = ema_path if cfg[\'checkpoint_selection\'] == \'ema\' else normal_path\\n    Path(cfg[\'checkpoint_manifest\']).write_text(str(selected_path.resolve()), encoding=\'utf-8\')\\n    summary = {\'normal\': str(normal_path.resolve()), \'ema\': str(ema_path.resolve()),\\n               \'selected\': str(selected_path.resolve()), \'selected_type\': cfg[\'checkpoint_selection\'],\\n               \'epochs\': self.curr_epoch, \'diffusion_steps\': cfg[\'diffusion_steps\'],\\n               \'ema_update\': \'per_batch\', \'ema_decay\': self.ema_decay}\\n    Path(cfg[\'models_manifest\']).write_text(json.dumps(summary, indent=2), encoding=\'utf-8\')\\n    print(\'Normal model checkpoint:\', normal_path)\\n    print(\'EMA model checkpoint:\', ema_path)\\n    print(\'Selected checkpoint for generation:\', selected_path)\\nTrainer.run_loop = train_and_save\\n\\nmode = sys.argv[2]\\nif mode != \'train\':\\n    raise RuntimeError(\'18 is training-only. Use notebook 19 to generate synthetic data.\')\\nif cfg[\'gpu\'] >= 0 and not torch.cuda.is_available():\\n    raise RuntimeError(\'CUDA unavailable. Set GPU=-1 for CPU, or fix CUDA environment.\')\\nargs = argparse.Namespace(dataname=cfg[\'dataname\'], mode=mode, method=\'tabdiff\',\\n    gpu=cfg[\'gpu\'], device=(\'cuda:%d\'%cfg[\'gpu\']) if cfg[\'gpu\']>=0 else \'cpu\',\\n    debug=False, no_wandb=True, exp_name=cfg[\'exp_name\'], deterministic=True,\\n    y_only=False, non_learnable_schedule=False, num_samples_to_generate=cfg[\'n_generate\'],\\n    ckpt_path=None, report=False, num_runs=1, impute=False)\\nmodule.main(args)\\n"\nRUNNER = OUT / "run_tabdiff.py"\nRUNNER.write_text(runner_source, encoding="utf-8")\nconfig = {"dataset_name": DATASET_NAME, "input_path": str(INPUT),\n    "input_sha256": INPUT_SHA256, "input_rows": INPUT_ROWS,\n    "dataset_conditions": dataset_entry["conditions"],\n    "aed_positive_value": dataset_entry["aed_positive_value"],\n    "repo": str(TABDIFF_REPO), "dataname": DATA_NAME, "exp_name": EXP_NAME,\n          "epochs": EPOCHS, "batch": TRAIN_BATCH_SIZE, "ema_decay": EMA_DECAY, "ema_update": "per_batch",\n          "sample_batch": SAMPLE_BATCH_SIZE, "diffusion_steps": DIFFUSION_STEPS,\n          "checkpoint_selection": CHECKPOINT_SELECTION,\n          "max_train_rows": MAX_TRAIN_ROWS, "seed": SEED,\n          "loss_eval_every": LOSS_EVAL_EVERY, "loss_eval_rows": LOSS_EVAL_ROWS,\n          "loss_eval_batch": LOSS_EVAL_BATCH_SIZE,\n          "history_path": str(OUT / "training_history.jsonl"),\n          "gpu": GPU, "n_generate": N_GENERATE, "commit": commit,\n          "checkpoint_manifest": str(OUT / "checkpoint.txt"),\n          "progress_path": str(OUT / "training_progress.json"),\n          "models_manifest": str(OUT / "models.json")}\nCONFIG = OUT / "run_config.json"\nCONFIG.write_text(json.dumps(config, indent=2), encoding="utf-8")\n', '# 学習を実行。経過時間・予想総所要時間と損失グラフを更新します。\nrun_command([TABDIFF_PYTHON, RUNNER, CONFIG, "train"], "train.log", show_timing=True)\n']
training_batch_results = []
_initial_training_out = OUT
_initial_training_dataset = DATASET_NAME
_initial_training_out_used = False

def train_one_dataset(dataset_name):
    global DATASET_NAME, INPUT, OUT, RUN_ID, DATA_NAME, MAX_TRAIN_ROWS, USE_ALL_ROWS
    global _initial_training_out_used
    if dataset_name not in DATASET_NAMES:
        raise ValueError(f"未知のデータセット: {dataset_name}")
    DATASET_NAME = dataset_name
    INPUT = PROCESSED_DIR / ("utstein_" + dataset_name.removeprefix("df_") + ".parquet")
    if not INPUT.is_file():
        raise FileNotFoundError(f"01で対象を保存してください: {INPUT}")
    # all指定は各データセットの件数で個別に解決します。
    MAX_TRAIN_ROWS = REQUESTED_TRAIN_ROWS
    USE_ALL_ROWS = REQUESTED_USE_ALL_ROWS
    if not _initial_training_out_used and dataset_name == _initial_training_dataset:
        OUT = _initial_training_out
        RUN_ID = OUT.name
        _initial_training_out_used = True
    else:
        RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        OUT = PROJECT / "artifacts/18_tabdiff" / RUN_ID
        OUT.mkdir(parents=True, exist_ok=False)
    DATA_NAME = "utstein_" + dataset_name.removeprefix("df_") + "_" + RUN_ID
    print(f"学習開始: {dataset_name} / 保存先: {OUT}")
    result = {"dataset":dataset_name,"output":str(OUT),"status":"running"}
    training_batch_results.append(result)
    try:
        for step_number, source in enumerate(TRAINING_STEPS, 1):
            exec(compile(source, f"<TabDiff {dataset_name} step {step_number}>", "exec"), globals())
        result.update(status="completed", rows=int(MAX_TRAIN_ROWS), dataname=DATA_NAME)
        print(f"学習完了: {dataset_name}")
    except BaseException as exc:
        result.update(status="interrupted" if isinstance(exc,KeyboardInterrupt) else "failed", error=str(exc))
        raise
    finally:
        (OUT / "dataset_training_status.json").write_text(
            json.dumps(result,ensure_ascii=False,indent=2),encoding="utf-8")
    return pd.DataFrame([result])

# 全モデルの入力ファイルの存在を、GPU学習開始前に確認します。
missing_inputs = [str(PROCESSED_DIR / ("utstein_"+name.removeprefix("df_")+".parquet"))
                  for name in DATASET_NAMES
                  if not (PROCESSED_DIR / ("utstein_"+name.removeprefix("df_")+".parquet")).is_file()]
if missing_inputs:
    raise FileNotFoundError("01の保存セルを実行してください。未保存の入力:\n"+"\n".join(missing_inputs))


### 1. df_main


In [ ]:
display(train_one_dataset("df_main"))


### 2. df_witness


In [ ]:
display(train_one_dataset("df_witness"))


### 3. df_bystandercpr


In [ ]:
display(train_one_dataset("df_bystandercpr"))


### 4. df_bystandercpr_ventilation


In [ ]:
display(train_one_dataset("df_bystandercpr_ventilation"))


### 5. df_bystandercpr_aed


In [ ]:
display(train_one_dataset("df_bystandercpr_aed"))


### 6. df_defibrillation


In [ ]:
display(train_one_dataset("df_defibrillation"))


### 7. df_airway


In [ ]:
display(train_one_dataset("df_airway"))


### 8. df_airway_airwaydevice


In [ ]:
display(train_one_dataset("df_airway_airwaydevice"))


### 9. df_adrenaline


In [ ]:
display(train_one_dataset("df_adrenaline"))


### 10. df_cardiac


In [ ]:
display(train_one_dataset("df_cardiac"))


### 11. df_noncardiac


In [ ]:
display(train_one_dataset("df_noncardiac"))


### 12. df_rosc


In [ ]:
display(train_one_dataset("df_rosc"))


### 13. df_survival


In [ ]:
display(train_one_dataset("df_survival"))


## 学習結果一覧


In [ ]:
display(pd.DataFrame(training_batch_results))


## 次の操作
19では、18で指定した症例数・学習条件を含む実行フォルダを選びます。モデル一覧から最終EMAまたは最終通常モデルを選びます（18のcheckpoint.txtの選択を19で自動強制しません）。20で生成結果を検証してください。

注意：症例数を減らすと学習時間は短くなりますが、少数の症例では稀なカテゴリーや欠損パターンを十分学習できないことがあります。複数の症例数で比較し、20の評価結果を確認してください。